# Advanced Java Learning: Latest Features & Best Practices
## Comprehensive Guide to Java 21 and Modern Development

This notebook covers the latest Java features, advanced concepts, and best practices for modern Java development.

## 1. Java Foundation & Setup

### Java Version Overview
- **Java 21 LTS** (September 2023) - Latest Long Term Support release
- Key features: Virtual Threads, Pattern Matching, Record Patterns, Sequenced Collections
- Regular releases every 6 months (Java 22, 23, 24, etc.)
- LTS releases every 3 years (Java 21, 25, etc.)

## 2. Records - Modern Data Classes (Java 16+)

### What are Records?
Records are a concise way to define immutable data carrier classes with automatic equals, hashCode, and toString.

```java
// Traditional class
public class Point {
    private final int x;
    private final int y;
    
    public Point(int x, int y) {
        this.x = x;
        this.y = y;
    }
    
    public int x() { return x; }
    public int y() { return y; }
    
    @Override
    public boolean equals(Object o) { ... }
    
    @Override
    public int hashCode() { ... }
    
    @Override
    public String toString() { ... }
}

// Record equivalent
public record Point(int x, int y) { }
```

### Advantages
- Less boilerplate code
- Immutable by design
- Automatic equals/hashCode/toString
- Better for DTOs and API models

## 3. Sealed Classes - Control Inheritance (Java 17+)

### What are Sealed Classes?
Sealed classes restrict which classes can extend them, enabling more maintainable and predictable hierarchies.

```java
public sealed class Shape permits Circle, Rectangle, Triangle {
    abstract double area();
}

public final class Circle extends Shape {
    private double radius;
    
    @Override
    double area() {
        return Math.PI * radius * radius;
    }
}

public sealed class Rectangle extends Shape permits Square {
    protected double width, height;
    
    @Override
    double area() {
        return width * height;
    }
}

public final class Square extends Rectangle {
    // Square implementation
}

public final class Triangle extends Shape {
    // Triangle implementation
}
```

### Use Cases
- Control extensibility and maintainability
- Use with pattern matching for exhaustive switches
- Better security and API design

## 4. Pattern Matching - Modern Control Flow (Java 21+)

### Type Patterns
```java
// Old way
if (obj instanceof String) {
    String str = (String) obj;
    System.out.println(str.length());
}

// Pattern matching (Java 16+)
if (obj instanceof String str) {
    System.out.println(str.length());
}
```

### Record Patterns (Java 21+)
```java
record Point(int x, int y) { }
record Circle(Point center, int radius) { }

// Destructuring with pattern matching
if (shape instanceof Circle(Point(var x, var y), var radius)) {
    System.out.println("Circle at (" + x + ", " + y + ") with radius " + radius);
}
```

### Switch Expressions with Patterns (Java 21+)
```java
String typeOf(Object obj) {
    return switch (obj) {
        case null -> "null";
        case Integer i && i > 0 -> "positive integer";
        case Integer i -> "non-positive integer";
        case String s -> "string of length " + s.length();
        case Point p -> "point at (" + p.x() + ", " + p.y() + ")";
        default -> "other";
    };
}
```

## 5. Virtual Threads - Lightweight Concurrency (Java 21+)

### What are Virtual Threads?
Virtual threads are lightweight threads managed by the JVM, enabling creation of millions of threads without overwhelming system resources.

### Traditional vs Virtual Threads
```java
// Platform threads (traditional)
Thread thread = new Thread(() -> {
    System.out.println("Running on platform thread");
});
thread.start();

// Virtual threads (Java 21+)
Thread vthread = Thread.ofVirtual()
    .name("virtual-", 0)
    .start(() -> {
        System.out.println("Running on virtual thread");
    });

// Or using ExecutorService
try (ExecutorService executor = Executors.newVirtualThreadPerTaskExecutor()) {
    for (int i = 0; i < 100_000; i++) {
        executor.submit(() -> blockingIO());
    }
}
```

### Benefits
- Handle millions of concurrent tasks
- Simplify asynchronous programming
- Lower memory footprint
- Better scalability for I/O-bound applications

## 6. Stream API & Functional Programming

### Intermediate Operations
```java
List<Integer> numbers = Arrays.asList(1, 2, 3, 4, 5, 6, 7, 8, 9, 10);

// filter, map, flatMap
List<Integer> result = numbers.stream()
    .filter(n -> n % 2 == 0)           // Keep even numbers
    .map(n -> n * n)                   // Square them
    .collect(Collectors.toList());

// flatMap for flattening
List<List<Integer>> lists = Arrays.asList(
    Arrays.asList(1, 2, 3),
    Arrays.asList(4, 5, 6)
);

List<Integer> flat = lists.stream()
    .flatMap(List::stream)
    .collect(Collectors.toList());
```

### Terminal Operations
```java
// collect, reduce, forEach, count, findFirst, anyMatch
int sum = numbers.stream()
    .reduce(0, Integer::sum);

Optional<Integer> first = numbers.stream()
    .filter(n -> n > 5)
    .findFirst();

boolean hasEven = numbers.stream()
    .anyMatch(n -> n % 2 == 0);
```

## 7. Sequenced Collections (Java 21+)

### New Collection Interfaces
Java 21 introduces sequenced collections with defined encounter order:

```java
// SequencedCollection interface methods
SequencedCollection<Integer> seq = new ArrayList<>(Arrays.asList(1, 2, 3, 4, 5));

// Get first/last elements
Integer first = seq.getFirst();      // NoSuchElementException if empty
Integer last = seq.getLast();

// Get first/last with safety
Optional<Integer> firstOpt = seq.stream().findFirst();

// Reverse order
SequencedCollection<Integer> reversed = seq.reversed();

// Remove first/last
Integer removed = seq.removeFirst();
Integer removedLast = seq.removeLast();
```

## 8. Modules - Java Module System

### What are Modules?
Modules provide encapsulation at a larger scale, organizing packages into named modules.

### Module Declaration (module-info.java)
```java
module com.example.myapp {
    requires java.base;              // Explicit dependency
    requires com.example.utils;      // Transitive dependency
    
    exports com.example.myapp.api;   // Public packages
    exports com.example.myapp.models;
    
    opens com.example.myapp.impl to
        java.base;
}
```

### Benefits
- Strong encapsulation
- Clear dependencies
- Reduced memory footprint
- Better maintainability for large projects

## 9. Exception Handling - Best Practices

### Custom Exceptions
```java
// Custom exception
public class InvalidTransactionException extends Exception {
    public InvalidTransactionException(String message) {
        super(message);
    }
    
    public InvalidTransactionException(String message, Throwable cause) {
        super(message, cause);
    }
}

// Using it
try {
    processTransaction(transaction);
} catch (InvalidTransactionException e) {
    logger.error("Transaction failed", e);
    throw new RuntimeException("Processing error", e);
}
```

### Try-With-Resources (Auto-Closing)
```java
try (FileReader reader = new FileReader("file.txt");
     BufferedReader br = new BufferedReader(reader)) {
    String line;
    while ((line = br.readLine()) != null) {
        System.out.println(line);
    }
} catch (IOException e) {
    logger.error("File reading failed", e);
}
```

## 10. Concurrency & Threading Updates

### Structured Concurrency (Preview in Java 21)
```java
// Coordinating concurrent tasks
try (var scope = new StructuredTaskScope.ShutdownOnFailure()) {
    Callable<String> task1 = scope.fork(() -> fetchUserData(userId));
    Callable<String> task2 = scope.fork(() -> fetchUserHistory(userId));
    
    scope.joinUntil(Instant.now().plusSeconds(5));
    
    String userData = task1.resultNow();
    String history = task2.resultNow();
} catch (InterruptedException e) {
    Thread.currentThread().interrupt();
}
```

### Thread-Safe Collections
```java
// Use concurrent collections for multi-threaded access
Map<String, Integer> concurrentMap = new ConcurrentHashMap<>();
List<String> concurrentList = Collections.synchronizedList(new ArrayList<>());

// Atomic variables
AtomicInteger counter = new AtomicInteger(0);
counter.incrementAndGet();
int currentValue = counter.get();
```

## 11. Generics & Type Erasure

### Generic Classes and Methods
```java
// Generic class
public class Container<T> {
    private T value;
    
    public void set(T value) {
        this.value = value;
    }
    
    public T get() {
        return value;
    }
}

// Bounded type parameters
public <T extends Number> double sum(T... numbers) {
    double sum = 0;
    for (T num : numbers) {
        sum += num.doubleValue();
    }
    return sum;
}

// Wildcard types
public void printList(List<?> list) {
    for (Object item : list) {
        System.out.println(item);
    }
}

public void addNumbers(List<? extends Number> list) {
    // Can only read
}
```

## 12. Reflection and Annotations

### Reflection API
```java
// Get class information
Class<?> clazz = String.class;
Method[] methods = clazz.getDeclaredMethods();
Field[] fields = clazz.getDeclaredFields();

// Invoke methods dynamically
Method method = clazz.getDeclaredMethod("length");
int length = (int) method.invoke("hello");

// Create instances dynamically
Constructor<?> constructor = clazz.getConstructor(String.class);
Object instance = constructor.newInstance("hello world");
```

### Custom Annotations
```java
@Target(ElementType.METHOD)
@Retention(RetentionPolicy.RUNTIME)
public @interface Deprecated {
    String value() default "";
    String since() default "";
}

@Deprecated("Use newMethod() instead")
public void oldMethod() {
    // implementation
}
```

## 13. Performance Optimization

### Memory Efficiency
```java
// Use primitive collections when possible
int[] primitiveArray = new int[1000];

// Boxing/Unboxing awareness
List<Integer> list = new ArrayList<>();  // Causes boxing overhead

// String optimization
String result = "";
for (int i = 0; i < 1000; i++) {
    result += i;  // BAD: Creates new String each iteration
}

// Use StringBuilder
StringBuilder sb = new StringBuilder();
for (int i = 0; i < 1000; i++) {
    sb.append(i);
}
String result = sb.toString();
```

### Profiling and JVM Flags
```bash
# Enable GC logging
java -Xmx2g -XX:+UseG1GC -XX:+PrintGCDetails -XX:+PrintGCDateStamps app.jar

# Use JProfiler or YourKit for detailed analysis
# Monitor heap size, GC behavior, and thread performance
```

## 14. Best Practices Summary

### Code Quality
1. **Use Records** for data classes instead of boilerplate POJOs
2. **Prefer sealed classes** to control inheritance hierarchies
3. **Use pattern matching** for cleaner control flow
4. **Leverage streams** for functional transformations
5. **Use virtual threads** for I/O-bound concurrent applications

### Architecture
1. **Implement SOLID principles** (Single Responsibility, Open/Closed, Liskov, Interface Segregation, Dependency Inversion)
2. **Use dependency injection** frameworks (Spring, Quarkus)
3. **Embrace modular design** with the Java Module System
4. **Separate concerns** with clear layering (Controller, Service, Repository)
5. **Use design patterns** appropriately (Builder, Factory, Singleton, Strategy)

### Testing
1. **Write unit tests** with JUnit 5
2. **Use mock frameworks** (Mockito)
3. **Aim for high code coverage** (>80%)
4. **Test edge cases** and error conditions
5. **Use test fixtures** for setup

### Performance
1. **Profile before optimizing**
2. **Monitor garbage collection**
3. **Use appropriate data structures**
4. **Batch operations** when possible
5. **Avoid premature optimization**

### Security
1. **Input validation** always
2. **Use prepared statements** for SQL
3. **Implement proper authentication/authorization**
4. **Sanitize sensitive data**
5. **Keep dependencies updated**

## 15. Modern Java Frameworks & Tools

### Popular Frameworks
- **Spring Boot** - Microservices and web applications
- **Quarkus** - Cloud-native and serverless applications
- **Jakarta EE** - Enterprise Java applications
- **Helidon** - Lightweight microservices framework

### Build Tools
- **Maven** - Dependency management and build automation
- **Gradle** - Flexible build automation

### Testing Frameworks
- **JUnit 5** - Modern testing framework
- **Mockito** - Mocking framework
- **TestNG** - Alternative testing framework
- **AssertJ** - Fluent assertions

### Monitoring & Logging
- **SLF4J + Logback/Log4J2** - Logging
- **Micrometer** - Metrics collection
- **OpenTelemetry** - Observability

## Resources & Next Steps

### Official Resources
- [Java Official Website](https://www.oracle.com/java/)
- [Java 21 Release Notes](https://www.oracle.com/java/21-relnotes/)
- [Java Language Specification](https://docs.oracle.com/javase/specs/)
- [JVM Language and Tool Specifications](https://docs.oracle.com/en/java/)

### Learning Paths
1. **Start with fundamentals** - Variables, control flow, OOP basics
2. **Master collections and generics** - Essential for real-world code
3. **Learn functional programming** - Streams and lambda expressions
4. **Explore concurrency** - Virtual threads and structured concurrency
5. **Practice with projects** - Build real applications
6. **Master a framework** - Spring Boot or Quarkus
7. **Learn DevOps basics** - Docker, Kubernetes, CI/CD

### Practice Projects
- Build a REST API with Spring Boot
- Create a microservices application
- Develop a multi-threaded application using virtual threads
- Build a command-line tool with picocli
- Create a data processing pipeline with streams